# Lab 05 · ReAct agents: reason, act, observe

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Act, observe, then answer

An agent can choose to use a tool, observe its result, and then finish. This tiny loop is limited to two decisions. The weather tool returns a local example; the model's actions may vary.

In [ ]:
weather = None
for turn in range(2):
    action = ask(f"Known weather: {weather}. Choose exactly LOOKUP or FINISH. "
                 "If weather is unknown, LOOKUP before FINISH.").strip().upper().strip(".")
    print("Action:", action)
    if action == "LOOKUP":
        weather = "rainy"  # Result of a local teaching tool.
        print("Observation:", weather)
    elif action == "FINISH" and weather is not None:
        print("Advice:", ask(f"Weather: {weather}. Suggest one indoor or outdoor activity."))
        break
    else:
        print("Invalid action or missing observation; stop for review.")
        break

**Next:** Apply the same action–observation loop to simulated dependency metrics.

## Inspect a dependency before choosing its resilience policy
**Start with a problem:** A fixed resilience answer may be wrong when a dependency is slow or failing for a different reason.

**In this lab:** Let the agent inspect simulated metrics before recommending a policy; bound its actions and check what it observed.

**So far:** Lab 04 let a model select one tool and our code checked it.

**New:** a bounded ReAct-style loop: choose an action, run an allowed tool, observe, then choose again. In checkout, a slow Inventory service can occupy threads in the services that wait for it, spreading the failure.

Our specific exercise configuration chooses a **circuit breaker** when repeated failures reach 50%; a bounded retry is considered only for transient failures on an idempotent call. The 50% threshold is illustrative, for this example; production settings require workload evidence. The agent recommends a policy; it does not deploy infrastructure.

The metrics are simulated teaching inputs. Inventory's 5-second value is a peak, not a measured p99; the failure rates and the 50% decision threshold are exercise assumptions. A retry also requires a transient failure and a bounded policy; idempotency alone is insufficient.

**At a glance**

![Lab 05 at a glance: Goal, then Next action (JSON), then Guards (known tool · facts read), then Run the tool (read target metrics), then Observation (added to history). Then back to Next action: repeat · at most 5 turns.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/05.svg)

## Think about the example

A slow dependency can occupy callers while they wait. Retrying adds more requests. A repeated card charge can have a second financial effect when the operation is not idempotent.

**Concepts:** Cascading failure; Circuit breaker vs retry; Idempotency and graceful degradation.

Use the scenario above to consider these questions before running the code:

1. Why can retries make a slow dependency worse?
2. Why must the non-idempotent Payments charge not be blindly retried?

## The idea in context

ReAct-style control flow alternates model actions with tool observations. The example exposes those exchanges without deploying a checkout system.

An agent loop selects its next step from observations; it still needs a bound and allowed tools.

| Key term | Meaning |
| --- | --- |
| **Agent** | A system in which a model chooses its next action in a loop, based on what it has observed. |
| **Observation** | The result of an action, fed back into the next prompt. |
| **History** | Every action and observation so far; the agent's view of the task. |
| **Guard** | A rule in code that the agent cannot override, such as a turn limit. |

## Pattern · One-Step Tool Agent

*Related to the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Connection** | Lab 04 used a small tool loop to look up, validate, and format data. Here the agent observes service metrics before suggesting a resilience pattern. |
| **Flow** | Goal → Choose action → Tool → Observe → repeat, or Finish |
| **Guard** | A loop can run forever or finish without reading metrics for the target. The turn limit and finish guard address both. |

## Main exercise at a glance

1. Prepare observations
2. Define the task and available actions
3. Choose, observe, and repeat
4. Check the recommendation and its limits

## Step 1 · Prepare observations
The agent can list the dependencies or read one record from this small local dataset. Unknown names produce an error. The next cells use ordinary `if` statements to select an action.

In [ ]:
import json

METRICS = {
    "Inventory": {"service": "Inventory", "failure_rate": 0.7, "peak_ms": 5000,
                  "idempotent": True, "transient_failure": True},
    "Payments": {"service": "Payments", "failure_rate": 0.01, "p99_ms": 1500,
                 "idempotent": False, "transient_failure": True},
}
print("Available dependencies:", list(METRICS))

## Step 2 · Define the task and available actions
The model can choose `list_dependencies`, `read_metrics`, or `finish`. The policy tells it to read the target's metrics before choosing. The following cell sends this policy and the observations on every turn.

In [ ]:
target = "Inventory"
POLICY = (
    "For this exercise: failure_rate >= 0.5 -> circuit breaker; "
    "otherwise idempotent AND transient_failure -> bounded retry; otherwise no automatic retry. "
    "Never infer metrics: read the target's record first."
)
FORMAT = 'JSON only: {"action":"tool name or finish","input":"service name or policy label"}'
print("Target:", target)

## Step 3 · Choose, observe, and repeat
On each turn, ask the model for an action, run the matching action with `if` statements, and append the observation. `finish` is allowed only after successfully reading the target's metrics. Five turns limit the loop; an unknown action becomes an error observation.

In [ ]:
history, decision, observed = [], None, False
for turn in range(1, 6):
    prompt = (f"{POLICY}\nGoal: recommend a resilience policy for {target}.\n"
              "Tools: list_dependencies (input ignored), read_metrics (exact service name).\n"
              f"History: {json.dumps(history)}\n{FORMAT}")
    step = json.loads(ask(prompt))
    if not isinstance(step, dict) or not isinstance(step.get("action"), str) or not isinstance(step.get("input"), str):
        raise ValueError("Action and input must be text.")
    action, value = step["action"], step["input"]
    if action == "finish":
        if not observed:
            raise ValueError("Read the target's metrics successfully before finishing.")
        decision = value.strip().lower()
        break
    if action == "list_dependencies":
        observation = list(METRICS)
    elif action == "read_metrics":
        observation = METRICS.get(value, {"error": "Unknown dependency; list exact names first."})
    else:
        observation = {"error": "Unknown action; use an allowed tool."}
    if action == "read_metrics" and isinstance(observation, dict) and observation.get("service") == target and "failure_rate" in observation:
        observed = True
    history.append({"action": action, "input": value, "observation": observation})
    print("Turn", turn, action, value, "-> observation:", observation)
assert decision is not None, "Turn budget exhausted; escalate instead of guessing."

## Step 4 · Check the recommendation and its limits
The independent checker uses the real target record. Choosing a breaker does not remove the need for timeouts, isolation and an appropriate fallback.

In [ ]:
facts = METRICS[target]
if facts["failure_rate"] >= 0.5:
    expected = "circuit breaker"
elif facts["idempotent"] and facts["transient_failure"]:
    expected = "bounded retry"
else:
    expected = "no automatic retry"

print("Recommendation:", decision)
print("Exercise expects:", expected, "| observations:", len(history))
assert decision == expected, "Recommendation fails the exercise policy."
print("Blindly retrying Payments can duplicate its non-idempotent charge.")

## What you should see
The model can list dependencies and read Inventory's simulated metrics before choosing a policy. An unknown name produces an error observation. Its actions can vary. A failed lookup or another service's metrics cannot unlock `finish`.

## Try it
1. Ask the agent to use `read_metrics` with `stock` before the loop. The error observation must not unlock `finish`.
2. In Step 2, change `target` to `"Payments"`, then inspect the resulting actions. Rerun Steps 2–4.
3. In Step 1, change Inventory's failure rate to 0.1. Rerun Step 1, then rerun Steps 2–4. Explain why both idempotency and a transient fault matter.
4. Pick a timeout, breaker and fallback for an optional Recommendations dependency that is slow or unavailable. Which user-visible feature degrades?

## Takeaway
An agent selects steps from observations, while code controls available actions, successful reads, budgets and final checks. Resilience patterns address different failure modes; indiscriminate retries can amplify an outage or duplicate a charge.